## Stage 03a — WYO Instrument Alignment

Cross-correlate WYO platform instruments against Picarro CH4 (trusted reference).
Writes lag-shifted Parquet to `03_instrument_aligned/`.

> Run `03_survey.ipynb` first to build `quality_manifest.yaml`.  Files on dates
> you marked `bad` are pre-rejected and auto-skipped in the widget.  Files on
> `uncertain` dates are shown with a `[?]` tag in the title.

> MML instruments (Ultra 321 + Pico 017 MML dates, LGR, Anem, GPS) are handled
> in `03b_align_mml.ipynb`.

| Section | Instrument | Primary ref | Secondary refs | Dates |
|---|---|---|---|---|
| A | WYO_aerisultra460 | Picarro | — | Feb 3–12 |
| B | LANL_aerisultra321 (WYO dates) | Picarro | Ultra460 aligned | Feb 3–12 |
| C | LANL_aerispico017 (WYO dates) | Picarro | Ultra460 + Ultra321 aligned | Feb 5–12 |

**Secondary refs** appear as dashed blue traces. They come from `03_instrument_aligned/`
(already lag-corrected) and are loaded lazily at the start of each section — run the
Apply cell after each section to make them available for the next.

**Outputs:** `lag_offsets_wyo.json`, `apply_manifest_wyo.json`,
aligned Parquet in `03_instrument_aligned/`.

**Workflow:** Imports → Config → Helpers → Load Picarro → Section A (auto + widget)
→ Apply A → Section B (auto + widget) → Apply B → Section C (auto + widget)
→ Save → Apply C → Pass-throughs.

In [ ]:
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, QUALITY_MANIFEST_PATH, REPO_ROOT
from src.provenance import git_info, check_clean, upstream_ref

display(HTML('<style>.plotly-graph-div { width: 100% !important; }</style>'))
print('Imports OK')

In [ ]:
CH4_COL   = 'CH4_ppm'
MAX_LAG_S = 1800

PICARRO_DIR  = STAGE_02_DIR / 'WYO_picarro'
ULTRA460_DIR = STAGE_02_DIR / 'WYO_aerisultra460' / 'Raw'
ULTRA321_DIR = STAGE_02_DIR / 'LANL_aerisultra321' / 'Raw'
PICO017_DIR  = STAGE_02_DIR / 'LANL_aerispico017'  / 'Raw'

with open(STAGE_02_DIR / 'routing_manifest.json') as _fh:
    ROUTING = json.load(_fh)
_mml = sum(1 for v in ROUTING.values() if v == 'MML')
_wyo = sum(1 for v in ROUTING.values() if v == 'WYO')
print(f'Routing manifest: {len(ROUTING)} entries  (MML={_mml}, WYO={_wyo})')
print('Config OK')

In [ ]:
from src.align import (
    resample_series, cross_correlate, raw_stem, date_tag,
    apply_lag_to_parquet, load_quality_manifest, file_quality,
    load_aligned_series, reference_bad_dates, resume_review,
)


def load_parquet_col(path, col):
    return pd.read_parquet(path, columns=[col])[col].dropna()


def load_all_series(parquet_dir, col='CH4_ppm', glob='*.parquet'):
    """Concat + resample all parquet files in parquet_dir into one long Series."""
    files = sorted(parquet_dir.glob(glob))
    if not files:
        return pd.Series(dtype=float)
    combined = pd.concat([load_parquet_col(f, col) for f in files]).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    return resample_series(combined)


def auto_correlate(test_files, ref_data, col='CH4_ppm'):
    suggestions = {}
    print(f"{'IDX':>4}  {'FILE':<55}  {'AUTO LAG':>10}")
    print('-' * 75)
    for i, f in enumerate(test_files):
        sig = resample_series(load_parquet_col(f, col))
        lag = cross_correlate(ref_data, sig)
        suggestions[f.stem] = lag
        print(f'[{i:>2}]  {f.name:<55}  {lag:>+10.1f}s')
    return suggestions


def save_lag_offsets_wyo():
    STAGE_03_DIR.mkdir(parents=True, exist_ok=True)
    g = globals()
    def _lags(conf, rej):
        return {k: v for k, v in conf.items() if k not in rej}
    git_hash, git_dirty = git_info(REPO_ROOT)
    manifest = {
        'stage':     '03a_align_wyo',
        'run_utc':   datetime.now(timezone.utc).isoformat(),
        'git_hash':  git_hash,
        'git_dirty': git_dirty,
        'upstream':  upstream_ref(STAGE_02_DIR / 'run_manifest.json'),
        'lags': {
            'WYO_aerisultra460':  _lags(g.get('u460_confirmed',     {}), g.get('u460_rejected',     set())),
            'LANL_aerisultra321': _lags(g.get('u321_wyo_confirmed', {}), g.get('u321_wyo_rejected', set())),
            'LANL_aerispico017':  _lags(g.get('pico_wyo_confirmed', {}), g.get('pico_wyo_rejected', set())),
        },
        'rejected': {
            'WYO_aerisultra460':  sorted(g.get('u460_rejected',     set())),
            'LANL_aerisultra321': sorted(g.get('u321_wyo_rejected', set())),
            'LANL_aerispico017':  sorted(g.get('pico_wyo_rejected', set())),
        },
    }
    with open(STAGE_03_DIR / 'lag_offsets_wyo.json', 'w') as fh:
        json.dump(manifest, fh, indent=2)


print('Helpers loaded.')

In [ ]:
def pre_reject_from_manifest(files, instrument, quality_manifest, cascade_bad_dates=None):
    """
    Return a set of file stems to pre-reject before the alignment widget.

    Two sources of pre-rejection:
    1. File itself is marked 'bad' in the quality manifest.
    2. Its date falls on a date where the alignment reference instrument was marked bad
       (cascade_bad_dates), meaning alignment is impossible regardless of data quality.

    The alignment widget auto-skips pre-rejected files; Commit overrides.
    """
    direct  = set()
    cascade = set()
    for f in files:
        status, _ = file_quality(quality_manifest, instrument, f)
        if status == 'bad':
            direct.add(f.stem)
        elif cascade_bad_dates and date_tag(f) in cascade_bad_dates:
            cascade.add(f.stem)
    if direct:
        print(f'  Pre-rejected (survey: bad) — {len(direct)} file(s):')
        for stem in sorted(direct): print(f'    {stem}')
    if cascade:
        print(f'  Pre-rejected (Picarro bad on date) — {len(cascade)} file(s):')
        for stem in sorted(cascade): print(f'    {stem}')
    return direct | cascade


def make_review_widget(
    ref_data, test_files, test_name, suggestions, confirmed, rejected,
    ref_name='Ref', save_fn=None, test_cols=None, normalize=False,
    secondary_refs=None, quality_tags=None, pre_rejected=None,
):
    """
    Interactive lag-review widget.

    ref_data      : pd.Series | dict[str, pd.Series]   Primary reference(s); gray dotted.
    secondary_refs: dict[str, pd.Series] | None         Already-aligned refs; dashed blue.
    quality_tags  : dict[stem, {status,reason}] | None  Survey tags for title annotation.
    pre_rejected  : set[str] | None                     Stems auto-skipped; Commit overrides.
    test_cols     : list[str] | None                    Columns from test files (default CH4_ppm).
    normalize     : bool                                Z-score all traces.
    """
    if not test_files:
        print(f'{test_name}: no files to review')
        return

    state      = {'idx': 0}
    _ref_dict  = ref_data if isinstance(ref_data, dict) else {ref_name: ref_data}
    _n_ref     = len(_ref_dict)
    _sec       = secondary_refs or {}
    _n_sec     = len(_sec)
    _test_cols = list(test_cols or ['CH4_ppm'])
    _qual      = quality_tags or {}
    _pre_rej   = set(pre_rejected or [])

    REF_COLORS  = ['#888888', '#AAAAAA', '#BBBBBB', '#CCCCCC']
    SEC_COLORS  = ['#5DADE2', '#85C1E9', '#7FB3D3', '#AED6F1']
    TEST_COLORS = ['#E67E22', '#2980B9', '#27AE60', '#8E44AD']

    fig = go.FigureWidget(layout=go.Layout(
        autosize=True, height=420,
        margin=dict(l=55, r=10, t=10, b=30),
        yaxis=dict(title='z-score' if normalize else _test_cols[0]),
        legend=dict(x=1.01, y=1, xanchor='left', font=dict(size=10)),
        hovermode='x unified',
    ))
    for i, rk in enumerate(_ref_dict.keys()):
        fig.add_scatter(name=rk,
                        line=dict(color=REF_COLORS[i % len(REF_COLORS)], width=1.5, dash='dot'),
                        opacity=0.85)
    for i, sk in enumerate(_sec.keys()):
        fig.add_scatter(name=f'{sk} ▸aligned',
                        line=dict(color=SEC_COLORS[i % len(SEC_COLORS)], width=1.5, dash='dash'),
                        opacity=0.80)
    for i, col in enumerate(_test_cols):
        fig.add_scatter(name=f'{test_name} — {col}',
                        line=dict(color=TEST_COLORS[i % len(TEST_COLORS)], width=2.5))

    title_html = widgets.HTML(value='')
    lag_slider = widgets.FloatSlider(
        value=0.0, min=-MAX_LAG_S, max=MAX_LAG_S, step=0.1,
        description='Lag (s):', continuous_update=True, readout_format='.1f',
        layout=widgets.Layout(width='100%'), style={'description_width': '60px'},
    )
    btn_prev   = widgets.Button(description='◀ Prev',        layout=widgets.Layout(width='88px'))
    btn_next   = widgets.Button(description='Next ▶',        layout=widgets.Layout(width='88px'))
    btn_commit = widgets.Button(description='Commit & Next', button_style='success',
                                layout=widgets.Layout(width='100%', height='34px'))
    btn_bad    = widgets.Button(description='Mark Bad & Next', button_style='danger',
                                layout=widgets.Layout(width='100%', height='34px'))

    # Column checkboxes — toggle which test traces are visible
    col_boxes = [
        widgets.Checkbox(value=True, description=col, indent=False,
                         layout=widgets.Layout(width='auto'),
                         style={'description_width': 'initial'})
        for col in _test_cols
    ]
    col_selector = widgets.VBox(col_boxes, layout=widgets.Layout(gap='2px'))

    # Xcorr column dropdown — pick which column drives the auto-lag suggestion.
    # Changing it recomputes cross-correlation for the current file on-the-fly
    # and snaps the slider; also updates suggestions[] so the result sticks.
    xcorr_drop = widgets.Dropdown(
        options=_test_cols, value=_test_cols[0],
        layout=widgets.Layout(width='100%'),
    )
    xcorr_lbl = widgets.HTML(value='<small style="color:#aaa">—</small>')

    log = widgets.Output(layout=widgets.Layout(
        max_height='80px', overflow_y='auto', border='1px solid #ddd', padding='4px',
    ))

    def _active_idxs():
        return {i for i, cb in enumerate(col_boxes) if cb.value}

    def _zscore(s):
        mu, sig = s.mean(), s.std()
        return (s - mu) / sig if sig > 0 else s * 0.0

    def _update_fig(idx, lag_s):
        active = _active_idxs()
        if idx >= len(test_files):
            with fig.batch_update():
                for trace in fig.data:
                    trace.x = []; trace.y = []
            title_html.value = (
                f'<b>{test_name} — complete</b>  '
                f'<span style="color:#888">({len(confirmed)} committed, {len(rejected)} rejected)</span>'
            )
            return

        f           = test_files[idx]
        key         = f.stem
        auto_lag    = suggestions.get(key, 0.0)
        qual_status = _qual.get(key, {}).get('status', '')

        try:
            test_dict = {col: resample_series(load_parquet_col(f, col)) for col in _test_cols}
        except Exception as e:
            title_html.value = f'<span style="color:red">ERROR loading {f.name}: {e}</span>'
            return

        first_sig = next(iter(test_dict.values()))
        t0 = first_sig.index[0]  - pd.Timedelta(hours=1)
        t1 = first_sig.index[-1] + pd.Timedelta(hours=1)

        with fig.batch_update():
            fig.layout.xaxis.autorange = True
            fig.layout.yaxis.autorange = True
            ti = 0
            for rk, rseries in _ref_dict.items():
                p = _zscore(rseries[t0:t1]) if normalize else rseries[t0:t1]
                fig.data[ti].x = p.index.tolist(); fig.data[ti].y = p.values.tolist()
                ti += 1
            for sk, sseries in _sec.items():
                p = _zscore(sseries[t0:t1]) if normalize else sseries[t0:t1]
                fig.data[ti].x = p.index.tolist(); fig.data[ti].y = p.values.tolist()
                ti += 1
            for i, (col, sig) in enumerate(test_dict.items()):
                p       = _zscore(sig) if normalize else sig
                shifted = (sig.index + pd.Timedelta(seconds=lag_s)).tolist()
                fig.data[ti].x       = shifted
                fig.data[ti].y       = p.values.tolist()
                fig.data[ti].name    = f'{col} ({lag_s:+.1f}s)'
                fig.data[ti].visible = i in active
                ti += 1

        status_tag = (
            f'  <span style="color:#1E8449">✓ {confirmed[key]:+.1f}s</span>' if key in confirmed else
            f'  <span style="color:#C0392B">✗ rejected</span>'               if key in rejected  else ''
        )
        pre_tag  = (' <span style="color:#C0392B">[PRE-BAD]</span>'
                    if key in _pre_rej and key not in confirmed else '')
        qual_tag = {'uncertain': ' <span style="color:#D35400">[?]</span>',
                    'bad':       ' <span style="color:#C0392B">[survey:bad]</span>'}.get(qual_status, '')
        n_ref_pts = int(next(iter(_ref_dict.values()))[t0:t1].notna().sum())
        dtag      = date_tag(f)
        subtitle  = (f'20{dtag[:2]}-{dtag[2:4]}-{dtag[4:6]}  '
                     f'{len(first_sig):,} rows  '
                     f'{first_sig.index[0].strftime("%H:%M")}–{first_sig.index[-1].strftime("%H:%M")} UTC  '
                     f'ref: {n_ref_pts:,} pts')
        if _sec:
            subtitle += '  sec: ' + ', '.join(f'{k}: {len(v[t0:t1]):,}' for k, v in _sec.items())
        title_html.value = (
            f'<b>[{idx+1}/{len(test_files)}]  {f.name}</b>'
            f'{pre_tag}{qual_tag}{status_tag}'
            f'<br><small style="color:#777">{subtitle}</small>'
        )
        xcorr_lbl.value = f'<small style="color:#888">auto ({xcorr_drop.value}): {auto_lag:+.1f}s</small>'

    def _recompute_xcorr():
        if state['idx'] >= len(test_files): return
        f   = test_files[state['idx']]
        col = xcorr_drop.value
        xcorr_lbl.value = '<small style="color:#aaa">computing…</small>'
        try:
            sig         = resample_series(load_parquet_col(f, col))
            primary_ref = next(iter(_ref_dict.values()))
            lag         = round(cross_correlate(primary_ref, sig), 1)
            suggestions[f.stem] = lag   # persist for this session
            lag_slider.value    = lag
            xcorr_lbl.value = f'<small style="color:#555">xcorr ({col}): {lag:+.1f}s → set</small>'
        except Exception as e:
            xcorr_lbl.value = f'<small style="color:red">xcorr error: {e}</small>'

    def go_to(idx):
        if 0 <= idx < len(test_files):
            key = test_files[idx].stem
            if key in _pre_rej and key not in confirmed:
                with log: print(f'auto-skip  {key}  (pre-rejected)')
                state['idx'] = idx + 1; go_to(state['idx']); return
            lag_slider.value = confirmed.get(key, suggestions.get(key, 0.0))
        _update_fig(idx, lag_slider.value)

    lag_slider.observe(lambda c: _update_fig(state['idx'], c['new']), names='value')
    xcorr_drop.observe(lambda c: _recompute_xcorr(),                  names='value')
    for cb in col_boxes:
        cb.observe(lambda c: _update_fig(state['idx'], lag_slider.value), names='value')

    def on_prev(_): state['idx'] = max(0, state['idx'] - 1); go_to(state['idx'])
    def on_next(_): state['idx'] += 1; go_to(state['idx'])

    def on_commit(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem
        lag = round(lag_slider.value, 1)
        confirmed[key] = lag; rejected.discard(key); _pre_rej.discard(key)
        if save_fn: save_fn()
        with log: print(f'COMMITTED  {key}  {lag:+.1f}s')
        state['idx'] += 1; go_to(state['idx'])

    def on_bad(_):
        if state['idx'] >= len(test_files): return
        key = test_files[state['idx']].stem
        rejected.add(key); confirmed.pop(key, None)
        if save_fn: save_fn()
        with log: print(f'REJECTED   {key}')
        state['idx'] += 1; go_to(state['idx'])

    btn_prev.on_click(on_prev); btn_next.on_click(on_next)
    btn_commit.on_click(on_commit); btn_bad.on_click(on_bad)

    _div    = lambda: widgets.HTML('<hr style="margin:6px 0;border:none;border-top:1px solid #ddd">')
    nav_row = widgets.HBox([btn_prev, btn_next], layout=widgets.Layout(gap='6px', margin='3px 0'))
    n_pre   = len(_pre_rej)
    pre_warn = [widgets.HTML(
        f'<small style="color:#C0392B">⚠ {n_pre} pre-rejected (Commit overrides)</small>'
    )] if n_pre else []

    left_panel = widgets.VBox(
        [nav_row, _div(),
         btn_commit, btn_bad, _div(),
         widgets.HTML('<small style="color:#666">Plot columns:</small>'),
         col_selector, _div(),
         widgets.HTML('<small style="color:#666">Xcorr column:</small>'),
         xcorr_drop, xcorr_lbl,
         *pre_warn, log],
        layout=widgets.Layout(width='270px', min_width='270px', padding='4px 14px 4px 4px'),
    )
    right_panel = widgets.VBox(
        [title_html, fig, lag_slider],
        layout=widgets.Layout(flex='1', min_width='0', width='100%'),
    )
    display(widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(width='100%', align_items='flex-start'),
    ))
    go_to(0)


print('Widget helper loaded.')


In [ ]:
quality_manifest = load_quality_manifest(QUALITY_MANIFEST_PATH)
n_tags = sum(len(v) for v in quality_manifest.values())
print(f'Quality manifest: {n_tags} entries loaded')
if not quality_manifest:
    print('  (run 03_survey.ipynb to build it — alignment will proceed without pre-rejection)')

# Dates where Picarro itself was marked bad — gas instruments on those dates
# cannot be aligned and will be pre-rejected automatically.
picarro_bad_dates = reference_bad_dates(quality_manifest, 'WYO_picarro', PICARRO_DIR)
if picarro_bad_dates:
    print(f'Picarro bad on {len(picarro_bad_dates)} date(s): {", ".join(sorted(picarro_bad_dates))}'
          f' — dependent files will be pre-rejected')
else:
    print('Picarro: no bad dates in manifest')

print('\nLoading Picarro reference...')
picarro_ref = load_all_series(PICARRO_DIR, col='CH4_ppm')
print(f'Picarro CH4_ppm: {len(picarro_ref):,} samples')
print(f'  Range: {picarro_ref.index[0]}  ->  {picarro_ref.index[-1]}')

## Resume saved review state

Off by default. Set `RESUME_REVIEW = True` only to continue an interrupted review after a kernel restart — never to carry lags across a timestamp change.

In [ ]:
# Resume saved review state — OFF by default.
#
# RESUME_REVIEW = False starts a clean review. That is what you want after an
# upstream timestamp change: Stage 01 now derives timestamps per row from the
# logger host clock, so every lag saved below was cross-correlated against the
# OLD timeline and is no longer valid.
#
# Flip to True ONLY to pick a review back up after a kernel restart, so a crash
# partway through does not cost the whole review.
#
# Why this exists at all: the widgets hold their state in notebook globals and
# save_lag_offsets_*() serialises whatever is in them. On a fresh kernel those are
# empty, so a top-to-bottom run would write an EMPTY manifest over a real one (the
# first Apply cell saves before the later sections have run) and then apply 0 s to
# every file. Seeding up front, here, is the only safe place to do it.
RESUME_REVIEW = False

if RESUME_REVIEW and 'u460_confirmed' not in dir():
    u460_confirmed, u460_rejected = resume_review(STAGE_03_DIR / 'lag_offsets_wyo.json', 'lags', 'WYO_aerisultra460')
if RESUME_REVIEW and 'u321_wyo_confirmed' not in dir():
    u321_wyo_confirmed, u321_wyo_rejected = resume_review(STAGE_03_DIR / 'lag_offsets_wyo.json', 'lags', 'LANL_aerisultra321')
if RESUME_REVIEW and 'pico_wyo_confirmed' not in dir():
    pico_wyo_confirmed, pico_wyo_rejected = resume_review(STAGE_03_DIR / 'lag_offsets_wyo.json', 'lags', 'LANL_aerispico017')

if not RESUME_REVIEW:
    print('RESUME_REVIEW = False — starting a clean review.')
    print('Move lag_offsets_wyo.json aside first if it still holds lags from the old timeline.')


---
## A — Ultra 460 vs Picarro

Ultra 460 is WYO-only (Feb 3–12). Primary reference: Picarro CH4.
No secondary references (this is the first section; nothing aligned yet).

In [ ]:
u460_files = sorted(ULTRA460_DIR.glob('*.parquet'))
print(f'Ultra 460: {len(u460_files)} files\n')
u460_suggestions = auto_correlate(u460_files, picarro_ref)

In [ ]:
if 'u460_confirmed' not in dir(): u460_confirmed = {}
if 'u460_rejected'  not in dir(): u460_rejected  = set()
u460_pre_rejected = pre_reject_from_manifest(
    u460_files, 'WYO_aerisultra460', quality_manifest,
    cascade_bad_dates=picarro_bad_dates,
)
u460_rejected |= u460_pre_rejected

make_review_widget(
    picarro_ref, u460_files, 'Ultra460',
    u460_suggestions, u460_confirmed, u460_rejected,
    ref_name='Picarro (ref)',
    save_fn=save_lag_offsets_wyo,
    quality_tags=quality_manifest.get('WYO_aerisultra460', {}),
    pre_rejected=u460_pre_rejected,
)

---
### Apply A — Ultra 460

Run after finishing the Ultra 460 widget.  Writes aligned files to `03_instrument_aligned/`
so they are available as a secondary reference in Section B.

In [ ]:
def apply_instrument(instrument, subdirs, lags, rejected_stems,
                     apply_spectra=True, wyo_only=False, lag_ref_name=None):
    src_inst = STAGE_02_DIR / instrument
    dst_inst = STAGE_03_DIR / instrument
    n_ok = n_bad = n_warn = n_skip = 0
    for subdir in subdirs:
        if not apply_spectra and subdir in ('Spectra', 'Spectralite'):
            print(f'  [SKIP spectra]  {instrument}/{subdir}')
            continue
        src_dir = src_inst / subdir if subdir else src_inst
        if not src_dir.exists():
            continue
        for path in sorted(src_dir.glob('*.parquet')):
            if wyo_only and ROUTING.get(raw_stem(path)) == 'MML':
                n_skip += 1
                continue
            rs       = raw_stem(path)
            dst_base = dst_inst / subdir if subdir else dst_inst
            if rs in rejected_stems:
                dst_path = dst_base / 'bad' / path.name
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, dst_path)
                print(f'  [BAD]  {path.name:<55}  -> bad/')
                n_bad += 1
                continue
            lag_s = lags.get(rs)
            if lag_s is None:
                print(f'  [WARN no lag]  {instrument}/{subdir or ""}/{path.name} — using 0s')
                lag_s = 0.0; n_warn += 1
            dst_path = dst_base / path.name
            rows = apply_lag_to_parquet(
                path, lag_s, dst_path,
                ts_status='picarro_aligned',
                lag_ref=lag_ref_name,
            )
            print(f'  [OK]  {path.name:<55}  {lag_s:>+6.1f}s  [{rows:,} rows]')
            n_ok += 1
    print(f'  -> {instrument}: aligned={n_ok}, bad={n_bad}, warn={n_warn}, skipped_mml={n_skip}')
    return {'ok': n_ok, 'bad': n_bad, 'warn': n_warn, 'skipped_mml': n_skip}

print('Apply helper loaded.')

In [ ]:
APPLY_SPECTRA = True

save_lag_offsets_wyo()
with open(STAGE_03_DIR / 'lag_offsets_wyo.json') as fh:
    saved_wyo = json.load(fh)

print(f'\n{"="*60}\n  WYO_aerisultra460\n{"="*60}')
apply_stats_a = apply_instrument(
    'WYO_aerisultra460', ['Raw', 'Eng', 'Spectralite'],
    saved_wyo['lags'].get('WYO_aerisultra460', {}),
    set(saved_wyo['rejected'].get('WYO_aerisultra460', [])),
    apply_spectra=APPLY_SPECTRA,
    lag_ref_name='WYO_picarro',
)
print('\nApply A complete.  Ultra 460 aligned files now available as secondary ref for Section B.')

---
## B — Ultra 321 WYO dates vs Picarro

Primary reference: Picarro CH4.  Secondary reference: Ultra 460 aligned output
(dashed blue) — available after Apply A runs.
MML-date files are skipped here; handled in `03b_align_mml.ipynb`.

In [ ]:
u321_all = sorted(ULTRA321_DIR.glob('*.parquet'))
u321_wyo = [f for f in u321_all if ROUTING.get(raw_stem(f)) != 'MML']
print(f'Ultra 321 total: {len(u321_all)} files')
print(f'Ultra 321 WYO-date files: {len(u321_wyo)}  (MML skipped)\n')
u321_wyo_suggestions = auto_correlate(u321_wyo, picarro_ref)

# Load Ultra 460 aligned as secondary reference
_u460_aligned = load_aligned_series(STAGE_03_DIR, 'WYO_aerisultra460', 'Raw', 'CH4_ppm')
if _u460_aligned is not None:
    _u460_aligned = resample_series(_u460_aligned)
    print(f'\nSecondary ref — Ultra460 aligned: {len(_u460_aligned):,} pts')
    sec_refs_b = {'Ultra460 aligned': _u460_aligned}
else:
    print('\nSecondary ref — Ultra460 not yet aligned (run Apply A first for secondary ref display)')
    sec_refs_b = {}

In [ ]:
if 'u321_wyo_confirmed' not in dir(): u321_wyo_confirmed = {}
if 'u321_wyo_rejected'  not in dir(): u321_wyo_rejected  = set()
u321_wyo_pre_rejected = pre_reject_from_manifest(
    u321_wyo, 'LANL_aerisultra321', quality_manifest,
    cascade_bad_dates=picarro_bad_dates,
)
u321_wyo_rejected |= u321_wyo_pre_rejected

make_review_widget(
    picarro_ref, u321_wyo, 'Ultra321-WYO',
    u321_wyo_suggestions, u321_wyo_confirmed, u321_wyo_rejected,
    ref_name='Picarro (ref)',
    save_fn=save_lag_offsets_wyo,
    secondary_refs=sec_refs_b,
    quality_tags=quality_manifest.get('LANL_aerisultra321', {}),
    pre_rejected=u321_wyo_pre_rejected,
)

---
### Apply B — Ultra 321 WYO dates

In [ ]:
save_lag_offsets_wyo()
with open(STAGE_03_DIR / 'lag_offsets_wyo.json') as fh:
    saved_wyo = json.load(fh)

print(f'\n{"="*60}\n  LANL_aerisultra321 (WYO dates only)\n{"="*60}')
apply_stats_b = apply_instrument(
    'LANL_aerisultra321', ['Raw', 'Eng', 'Spectra'],
    saved_wyo['lags'].get('LANL_aerisultra321', {}),
    set(saved_wyo['rejected'].get('LANL_aerisultra321', [])),
    apply_spectra=APPLY_SPECTRA, wyo_only=True,
    lag_ref_name='WYO_picarro',
)
print('\nApply B complete.  Ultra 321 aligned files now available as secondary ref for Section C.')

---
## C — Pico 017 WYO dates vs Picarro

Primary reference: Picarro CH4.  Secondary references: Ultra 460 aligned + Ultra 321
aligned (both dashed blue family) — available after Apply A and Apply B run.
MML dates handled in `03b_align_mml.ipynb`.

In [ ]:
pico_all = sorted(PICO017_DIR.glob('*.parquet'))
pico_wyo = [f for f in pico_all if ROUTING.get(raw_stem(f)) != 'MML']
print(f'Pico 017 total: {len(pico_all)} files')
print(f'Pico 017 WYO-date files: {len(pico_wyo)}  (MML skipped)\n')
pico_wyo_suggestions = auto_correlate(pico_wyo, picarro_ref)

# Load secondary refs (Ultra460 + Ultra321 aligned outputs)
_u460_aligned = load_aligned_series(STAGE_03_DIR, 'WYO_aerisultra460', 'Raw', 'CH4_ppm')
_u321_aligned = load_aligned_series(STAGE_03_DIR, 'LANL_aerisultra321', 'Raw', 'CH4_ppm')
sec_refs_c = {}
if _u460_aligned is not None:
    sec_refs_c['Ultra460 aligned'] = resample_series(_u460_aligned)
    print(f'Secondary ref — Ultra460 aligned: {len(sec_refs_c["Ultra460 aligned"]):,} pts')
else:
    print('Secondary ref — Ultra460 not available (run Apply A)')
if _u321_aligned is not None:
    sec_refs_c['Ultra321 aligned'] = resample_series(_u321_aligned)
    print(f'Secondary ref — Ultra321 aligned: {len(sec_refs_c["Ultra321 aligned"]):,} pts')
else:
    print('Secondary ref — Ultra321 not available (run Apply B)')

In [ ]:
if 'pico_wyo_confirmed' not in dir(): pico_wyo_confirmed = {}
if 'pico_wyo_rejected'  not in dir(): pico_wyo_rejected  = set()
pico_wyo_pre_rejected = pre_reject_from_manifest(
    pico_wyo, 'LANL_aerispico017', quality_manifest,
    cascade_bad_dates=picarro_bad_dates,
)
pico_wyo_rejected |= pico_wyo_pre_rejected

make_review_widget(
    picarro_ref, pico_wyo, 'Pico017-WYO',
    pico_wyo_suggestions, pico_wyo_confirmed, pico_wyo_rejected,
    ref_name='Picarro (ref)',
    save_fn=save_lag_offsets_wyo,
    secondary_refs=sec_refs_c,
    quality_tags=quality_manifest.get('LANL_aerispico017', {}),
    pre_rejected=pico_wyo_pre_rejected,
)

---
## Save lag_offsets_wyo.json

Run once all widget sections are complete.  (Also auto-saved after every
Commit / Mark Bad click — no work is lost if the kernel dies.)

In [ ]:
save_lag_offsets_wyo()
lag_path = STAGE_03_DIR / 'lag_offsets_wyo.json'
print(f'Saved -> {lag_path}\n')
with open(lag_path) as fh:
    saved_wyo = json.load(fh)
for inst, lags in saved_wyo['lags'].items():
    rej = saved_wyo['rejected'].get(inst, [])
    if lags or rej:
        print(f'{inst}: {len(lags)} confirmed, {len(rej)} rejected')
        for stem, lag in sorted(lags.items()):
            print(f'  {stem:<55}  {lag:>+6.1f}s')

---
## Apply C — Pico 017 WYO dates

Then continue to the pass-through cells.

In [ ]:
APPLY_SPECTRA = True

with open(STAGE_03_DIR / 'lag_offsets_wyo.json') as fh:
    saved_wyo = json.load(fh)

apply_stats = {}

# ── Apply C: Pico 017 WYO ────────────────────────────────────────────────────
print(f'\n{"="*60}\n  LANL_aerispico017 (WYO dates only)\n{"="*60}')
apply_stats['LANL_aerispico017'] = apply_instrument(
    'LANL_aerispico017', ['Raw', 'Eng', 'Spectra'],
    saved_wyo['lags'].get('LANL_aerispico017', {}),
    set(saved_wyo['rejected'].get('LANL_aerispico017', [])),
    apply_spectra=APPLY_SPECTRA, wyo_only=True,
    lag_ref_name='WYO_picarro',
)

check_clean(REPO_ROOT, context='03a_apply_wyo')
regen_git_hash, regen_git_dirty = git_info(REPO_ROOT)
apply_manifest = {
    'stage':           '03a_apply_wyo',
    'run_utc':         datetime.now(timezone.utc).isoformat(),
    'git_hash':        saved_wyo['git_hash'],    # when the alignment was DECIDED
    'git_dirty':       saved_wyo['git_dirty'],
    'regen_git_hash':  regen_git_hash,      # when this output was last (re)built
    'regen_git_dirty': regen_git_dirty,
    'apply_spectra': APPLY_SPECTRA,
    'instruments':   {
        'WYO_aerisultra460':  apply_stats_a if 'apply_stats_a' in dir() else {},
        'LANL_aerisultra321': apply_stats_b if 'apply_stats_b' in dir() else {},
        'LANL_aerispico017':  apply_stats['LANL_aerispico017'],
    },
}
apply_path = STAGE_03_DIR / 'apply_manifest_wyo.json'
with open(apply_path, 'w') as fh:
    json.dump(apply_manifest, fh, indent=2)
print(f'\nApply manifest -> {apply_path}')
print('Run pass-through cells below to finish Stage 03a.')

---
## Pass-through: no_coverage → bad_timestamp

Stage 02 `no_coverage/` files have Mountain Time clocks — nothing Stage 03 can do.
Copied to `bad_timestamp/` so Stage 03 is a complete, self-contained dataset.

In [ ]:
NO_COVERAGE_SUBDIRS = {
    'LANL_aerisultra321': ['Raw', 'Eng', 'Spectra'],
    'LANL_aerispico017':  ['Raw', 'Eng', 'Spectra'],
}
passthrough_stats = {}
for inst, subdirs in NO_COVERAGE_SUBDIRS.items():
    n_ok = 0
    for subdir in subdirs:
        if not APPLY_SPECTRA and subdir == 'Spectra':
            continue
        src_dir = STAGE_02_DIR / inst / subdir / 'no_coverage'
        dst_dir = STAGE_03_DIR / inst / subdir / 'bad_timestamp'
        if not src_dir.exists():
            continue
        files = sorted(src_dir.glob('*.parquet'))
        if not files:
            continue
        dst_dir.mkdir(parents=True, exist_ok=True)
        for path in files:
            shutil.copy2(path, dst_dir / path.name)
            print(f'  [PASS]  {inst}/{subdir}/bad_timestamp/{path.name}')
            n_ok += 1
    passthrough_stats[inst] = {'copied': n_ok}
    print(f'  -> {inst}: {n_ok} files -> bad_timestamp/')

apply_path = STAGE_03_DIR / 'apply_manifest_wyo.json'
with open(apply_path) as fh:
    m = json.load(fh)
m['passthrough'] = passthrough_stats
with open(apply_path, 'w') as fh:
    json.dump(m, fh, indent=2)
print(f'\nno_coverage pass-through complete.')

---
## Pass-through: trusted instruments

Picarro and Sprinter carry trusted UTC timestamps. Copied from Stage 02 unchanged
so Stage 03 is a complete, self-contained aligned dataset.

In [ ]:
TRUSTED_INSTRUMENTS = ['WYO_picarro', 'WYO_sprinter']
trusted_stats = {}
for inst in TRUSTED_INSTRUMENTS:
    src_dir = STAGE_02_DIR / inst
    dst_dir = STAGE_03_DIR / inst
    files   = sorted(src_dir.glob('*.parquet'))
    if not files:
        print(f'[WARN]  {inst} — no parquet files in {src_dir}')
        trusted_stats[inst] = {'ok': 0}
        continue
    print(f'\n{"="*60}\n  {inst}  ({len(files)} files)\n{"="*60}')
    n_ok = 0
    for f in files:
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst_dir / f.name)
        print(f'  OK  {f.name}')
        n_ok += 1
    trusted_stats[inst] = {'ok': n_ok}

apply_path = STAGE_03_DIR / 'apply_manifest_wyo.json'
with open(apply_path) as fh:
    m = json.load(fh)
m['trusted'] = trusted_stats
with open(apply_path, 'w') as fh:
    json.dump(m, fh, indent=2)
print(f'\nTrusted pass-through complete.')
print(f'Stage 03a complete -> {STAGE_03_DIR}')